## 14.1 Setup:

In [20]:
# 14.1 Setup: retriever (rebuilds the vector DB once when chunks.jsonl changed) + corrective layer
import re
import time
import pandas as pd
from IPython.display import display
from src.paths import CHUNKS
from src.retriever import Retriever
from src.query import SpellFixer
from src.corrective import Corrective

ROOT = CHUNKS.parents[2]
EVAL = ROOT / "data" / "eval" / "benchmark"
MANUAL = EVAL / "manual_questions_with_answers.xlsx"
SET_C = EVAL / "set_C_claude100.csv"
ANSWERS = EVAL / "results_manual_v2" / "manual_review.csv"     # optional: saved answers (Step 0 + LLM checkpoint)
TRACE_OUT = EVAL / "trace_v1.csv"
EXPECTED_CHUNKS = 427                                          # STC chunks.jsonl; change when the index changes

for path in (MANUAL, SET_C):
    if not path.exists():
        raise FileNotFoundError(f"missing {path.name} → copy it into {EVAL}")

start = time.time()
R = Retriever()
CR = Corrective(R, SpellFixer(R.chunks))
print(f"{len(R.chunks)} chunks · retriever + vector DB ready in {time.time() - start:.0f} s")
assert len(R.chunks) == EXPECTED_CHUNKS, f"expected {EXPECTED_CHUNKS} chunks - wrong chunks.jsonl?"
assert "T102-1-01" not in R.row_of, "old table chunks are still loaded"

427 chunks · retriever + vector DB ready in 4 s


## 14.2 Fact checker: 

In [21]:
# 14.2 Fact checker: share of the expected answer's facts found in a text (same rules as 13.10, numbers fixed)
STOP_WORDS = set("""the a an and or of to in on for by with from as at is are be this that these those it its which what who
will can may should must also each every other than then into per not no any all their there here such has have had
was were been being if when where how does do done only more most some your you them they given give gives""".split())

def numbers(text):
    joined = re.sub(r"(?<=\d)[\s\u00a0\u2009\u202f](?=\d{3}\b)", "", str(text))
    return {found.replace(",", "") for found in re.findall(r"\d[\d,]*(?:\.\d+)?", joined) if len(found.replace(",", "")) >= 2}

def key_words(text):
    return {word[:6] for word in re.findall(r"[a-z]+", str(text).lower()) if len(word) >= 4 and word not in STOP_WORDS}

def facts(expected):
    return [piece.strip() for piece in re.split(r";|:|\.\s|\s-\s|\(\d\)", str(expected)) if key_words(piece) or numbers(piece)]

def fact_found(fact, text_words, text_numbers):
    fact_words = key_words(fact)
    words_ok = len(fact_words & text_words) >= max(1, len(fact_words) / 2) if fact_words else True
    return words_ok and numbers(fact) <= text_numbers

def coverage(expected, text):
    expected_facts, text_words, text_numbers = facts(expected), key_words(text), numbers(text)
    return round(sum(fact_found(fact, text_words, text_numbers) for fact in expected_facts) / max(len(expected_facts), 1), 2)

## 14.3 Step 0

In [22]:
# 14.3 Step 0: re-score the saved manual answers (no LLM) and show NHM23
def verdict(row):
    if row.action in ("reject", "scoped"):
        return "refused_ok" if str(row.what_to_expect).startswith("EXPECT 'Not found'") else "refused_wrong"
    if "not found" in str(row.answer).lower()[:60]:
        return "said_not_found"
    return "correct" if row.coverage >= 0.6 else "partial" if row.coverage >= 0.3 else "incorrect"

if not ANSWERS.exists():
    raise FileNotFoundError(f"put the Kaggle file manual_review.csv into {ANSWERS.parent}")
saved = pd.read_csv(ANSWERS)
saved["coverage"] = [coverage(expected, answer) for expected, answer in zip(saved.expected_answer, saved.answer)]
saved["auto_verdict"] = saved.apply(verdict, axis=1)
display(pd.crosstab(saved.model, saved.auto_verdict, margins=True))
display(saved.pivot(index="qid", columns="model", values=["auto_verdict", "coverage"]))
for row in saved[saved.qid == "NHM23"].itertuples():
    print(f"--- {row.model} (coverage {row.coverage})\n{row.answer}\n")

auto_verdict,correct,incorrect,partial,refused_ok,refused_wrong,said_not_found,All
model,,,,,,,
groq:openai/gpt-oss-120b,3,5,6,2,5,2,23
qwen2.5:7b,1,10,4,2,5,1,23
All,4,15,10,4,10,3,46


auto_verdict                                 coverage  \
model groq:openai/gpt-oss-120b      qwen2.5:7b groq:openai/gpt-oss-120b   
qid                                                                       
NHM01                  partial         partial                     0.36   
NHM02                  partial         partial                     0.45   
NHM03            refused_wrong   refused_wrong                      0.0   
NHM04                  correct         correct                     0.75   
NHM05                incorrect       incorrect                     0.06   
NHM06                  partial       incorrect                     0.33   
NHM07               refused_ok      refused_ok                      0.0   
NHM08               refused_ok      refused_ok                      0.0   
NHM09                incorrect       incorrect                      0.0   
NHM10            refused_wrong   refused_wrong                      0.0   
NHM11                  partial       incorrect                     0.35   
NHM12                incorrect       incorrect                     0.18   
NHM13                  correct         partial                     0.73   
NHM14           said_not_found  said_not_found                      0.0   
NHM15            refused_wrong   refused_wrong                     0.14   
NHM16            refused_wrong   refused_wrong                      0.0   
NHM17                  partial       incorrect                     0.39   
NHM18                incorrect       incorrect                      0.0   
NHM19                incorrect       incorrect                     0.09   
NHM20                  correct         partial                      1.0   
NHM21           said_not_found       incorrect                      0.0   
NHM22            refused_wrong   refused_wrong                      0.0   
NHM23                  partial       incorrect                     0.38   

                  
model qwen2.5:7b  
qid               
NHM01       0.36  
NHM02       0.45  
NHM03        0.0  
NHM04       0.75  
NHM05       0.19  
NHM06       0.17  
NHM07        0.0  
NHM08        0.0  
NHM09        0.0  
NHM10        0.0  
NHM11       0.29  
NHM12       0.12  
NHM13       0.36  
NHM14        0.0  
NHM15       0.14  
NHM16        0.0  
NHM17       0.22  
NHM18       0.11  
NHM19       0.09  
NHM20       0.44  
NHM21        0.0  
NHM22        0.0  
NHM23       0.25

--- groq:openai/gpt-oss-120b (coverage 0.38)
**Example of determining entry‑level salary for the position of Consultant (based on experience and qualification)**  

| Years of Experience | Qualification | Entry‑level Salary (₹) |
|---------------------|--------------|------------------------|
| 1 year | Medical Graduate with PG | 47 000 |
| 1 year | Full‑time Masters/PG in Health/Rural/Hospital/Other Management Sciences | 36 000 |
| 1 year | Full‑time PG/Masters in Science/Social Science background | 31 000 |
| 2 years | Medical Graduate with PG | 49 000 |
| 2 years | Full‑time Masters/PG in Health/Rural/Hospital/Other Management Sciences | 37 000 |
| 2 years | Full‑time PG/Masters in Science/Social Science background | 32 000 |
| 3 years | Medical Graduate with PG | 51 000 |
| 3 years | Full‑time Masters/PG in Health/Rural/Hospital/Other Management Sciences | 38 000 |
| 3 years | Full‑time PG/Masters in Science/Social Science background | 33 000 |
| 4 years | Medical Graduate with PG 

## 14.4 Questions to trace

In [23]:
# 14.4 Questions to trace: manual 23 + set C answerable (52), with their gold chunks
def gold_rows(gold):
    """Rows of R.chunks holding the gold evidence. Old table IDs (T101-1-14, T102-1-*) are matched by page."""
    rows = set()
    for item in str(gold).split(";"):
        item = item.strip()
        if item in R.row_of:
            rows.add(R.row_of[item])
        elif re.fullmatch(r"T\d+-\d+-(\d+|\*)", item):
            page = int(re.match(r"T(\d+)", item)[1])
            rows |= {i for i, chunk in enumerate(R.chunks) if chunk["doc_type"] == "table" and page in chunk["pdf_pages"]}
    return rows

def content_of(rows):
    kinds = {R.chunks[i]["doc_type"] for i in rows}
    if not rows:
        return "not_indexed"
    return "letter" if any(kind.startswith("letter") for kind in kinds) else "table" if "table" in kinds else "section"

manual = pd.read_excel(MANUAL)
manual.columns = manual.columns.str.strip()
set_c = pd.read_csv(SET_C)
set_c = set_c[set_c.expected == "answer"]
QUESTIONS = pd.concat([
    pd.DataFrame({"source": "manual", "qid": manual.qid, "question": manual.question, "qtype": manual.qtype,
                  "gold": manual.gold_chunks, "expected": manual.expected_answer}),
    pd.DataFrame({"source": "set_C", "qid": set_c.qid, "question": set_c.question, "qtype": set_c.qtype,
                  "gold": set_c.gold_chunks, "expected": set_c.reference_answer}),
], ignore_index=True)
QUESTIONS["rows"] = QUESTIONS.gold.map(gold_rows)
QUESTIONS["content"] = QUESTIONS.rows.map(content_of)
display(pd.crosstab(QUESTIONS.content, QUESTIONS.source, margins=True))

source,manual,set_C,All
content,,,
letter,3,15,18
not_indexed,2,0,2
section,11,32,43
table,7,5,12
All,23,52,75


## 14.5 Trace:

In [24]:
# 14.5 Trace: the first checkpoint each question fails (no LLM call; saved answers used for the last checkpoint)
POOL = 20            # ranks are looked up in each method's top 20
CHUNK_OK = 0.5       # the gold chunks must hold at least half of the expected facts
CONTEXT_OK = 0.5     # the final prompt (and the answer) must still hold at least half of them
MODEL = "groq:openai/gpt-oss-120b"
CHECKPOINTS = ["index", "chunk", "retrieval", "gate", "context", "llm", "reached_llm", "ok"]

def best_rank(question, method, rows):
    ranked = [row for row, score in R.ranked(question, POOL, method)]
    return next((rank for rank, row in enumerate(ranked, 1) if row in rows), None)

def trace_one(item, answers):
    prep = CR.prepare(item.question)
    ranks = {method: best_rank(prep["fixed"], method, item.rows) if item.rows else None for method in R.METHODS}
    gold_cov = coverage(item.expected, " ".join(R.chunks[row]["text"] for row in item.rows))
    in_final = any(R.row_of.get(hit["chunk_id"]) in item.rows for hit in prep["hits"])
    prompt_cov = coverage(item.expected, prep["prompt"]) if prep["prompt"] else 0.0
    answer = answers.get(item.qid)
    answer_cov = coverage(item.expected, answer) if answer is not None else None
    checks = [("index", bool(item.rows)),
              ("chunk", gold_cov >= CHUNK_OK),
              ("retrieval", ranks["rerank"] is not None and ranks["rerank"] <= 5),
              ("gate", prep["action"] in ("answer", "answer_with_note")),
              ("context", in_final and prompt_cov >= CONTEXT_OK),
              ("llm", answer_cov is None or answer_cov >= CONTEXT_OK)]
    lost_at = next((name for name, passed in checks if not passed), "ok" if answer_cov is not None else "reached_llm")
    return {"source": item.source, "qid": item.qid, "content": item.content, "qtype": item.qtype, "lost_at": lost_at,
            "gold_cov": gold_cov, **ranks, "grade": f"{prep['first']}→{prep['final']}", "best": prep["best"],
            "action": prep["action"], "in_final": in_final, "prompt_cov": prompt_cov, "answer_cov": answer_cov,
            "fixed": prep["fixed"], "final_hits": prep["hit_ids"]}

saved_answers = pd.read_csv(ANSWERS) if ANSWERS.exists() else pd.DataFrame(columns=["qid", "model", "answer"])
answers = dict(zip(saved_answers[saved_answers.model == MODEL].qid, saved_answers[saved_answers.model == MODEL].answer))
start = time.time()
TRACE = pd.DataFrame([trace_one(item, answers) for item in QUESTIONS.itertuples()])
TRACE["lost_at"] = pd.Categorical(TRACE.lost_at, CHECKPOINTS)
TRACE.to_csv(TRACE_OUT, index=False)
print(f"{len(TRACE)} questions traced in {time.time() - start:.0f} s → {TRACE_OUT.name}")

d:\SwasthyaSetu\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


75 questions traced in 322 s → trace_v1.csv


## 14.5a Try ONE question first

In [25]:
# 14.5a Try ONE question first: any error shows here, with the full traceback
first = next(QUESTIONS.itertuples())
display(pd.Series(trace_one(first, answers)))

source                                            manual
qid                                                NHM01
content                                            table
qtype                                        form / list
lost_at                                              llm
gold_cov                                            0.93
bm25                                                   1
vector                                                 1
hybrid                                                 1
rerank                                                 1
grade                                                S→S
best                                                1.17
action                                            answer
in_final                                            True
prompt_cov                                          0.57
answer_cov                                          0.36
fixed         Can you  give me  sample of appraisal form
final_hits       S079-13;S079-0

## 14.5b Trace all questions

In [26]:
# 14.5b Trace all questions; a crash names the question it happened on
start = time.time()
records = []
for item in QUESTIONS.itertuples():
    try:
        records.append(trace_one(item, answers))
    except Exception as error:
        raise RuntimeError(f"trace failed at {item.source} {item.qid}: {item.question[:60]}") from error
TRACE = pd.DataFrame(records)
TRACE["lost_at"] = pd.Categorical(TRACE.lost_at, CHECKPOINTS)
TRACE.to_csv(TRACE_OUT, index=False)
print(f"{len(TRACE)} questions traced in {time.time() - start:.0f} s → {TRACE_OUT.name}")

75 questions traced in 298 s → trace_v1.csv


In [27]:
# 14.6 The error map
display(pd.crosstab(TRACE.lost_at, TRACE.source, margins=True, dropna=False))
display(pd.crosstab(TRACE.lost_at, TRACE.content, margins=True, dropna=False))
display(TRACE.drop(columns=["fixed", "final_hits"]).sort_values(["lost_at", "source", "qid"]))

source,manual,set_C,All
lost_at,,,
index,2,0,2
chunk,1,2,3
retrieval,1,2,3
gate,5,2,7
context,3,0,3
llm,8,0,8
reached_llm,0,46,46
ok,3,0,3
All,23,52,75


content,letter,not_indexed,section,table,All
lost_at,,,,,
index,0,2,0,0,2
chunk,1,0,1,1,3
retrieval,2,0,1,0,3
gate,0,0,4,3,7
context,0,0,3,0,3
llm,1,0,4,3,8
reached_llm,13,0,30,3,46
ok,1,0,0,2,3
All,18,2,43,12,75


,source,qid,content,qtype,lost_at,gold_cov,bm25,vector,hybrid,rerank,grade,best,action,in_final,prompt_cov,answer_cov
6,manual,NHM07,not_indexed,coverage gap,index,0.00,NaN,NaN,NaN,NaN,I→I,-10.87,reject,False,0.00,0.00
7,manual,NHM08,not_indexed,coverage gap,index,0.00,NaN,NaN,NaN,NaN,I→I,-11.16,reject,False,0.00,0.00
17,manual,NHM18,section,list,chunk,0.05,5.0,5.0,4.0,3.0,S→S,3.14,answer,True,0.37,0.00
56,set_C,C-034,table,calculation,chunk,0.00,2.0,1.0,1.0,1.0,S→S,6.20,answer,True,0.50,NaN
70,set_C,C-048,letter,calculation,chunk,0.00,1.0,1.0,1.0,1.0,P→P,5.94,answer_with_note,True,1.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,set_C,C-051,letter,scenario,reached_llm,1.00,1.0,1.0,1.0,1.0,S→S,2.00,answer,True,1.00,NaN
74,set_C,C-052,table,calculation,reached_llm,1.00,1.0,2.0,1.0,1.0,P→P,5.96,answer_with_note,True,1.00,NaN
3,manual,NHM04,table,process,ok,1.00,2.0,1.0,1.0,1.0,S→S,7.95,answer,True,0.83,0.75
12,manual,NHM13,table,process,ok,0.91,1.0,1.0,1.0,1.0,S→S,1.03,answer,True,0.64,0.73


In [28]:
pd.set_option("display.max_rows", 100, "display.max_colwidth", 80)
display(TRACE[TRACE.lost_at.isin(["chunk", "retrieval", "gate", "context", "llm"])].drop(columns=["final_hits"]))

,source,qid,content,qtype,lost_at,gold_cov,bm25,vector,hybrid,rerank,grade,best,action,in_final,prompt_cov,answer_cov,fixed
0,manual,NHM01,table,form / list,llm,0.93,1.0,1.0,1.0,1.0,S→S,1.17,answer,True,0.57,0.36,Can you give me sample of appraisal form
1,manual,NHM02,section,fact,llm,0.82,1.0,1.0,1.0,1.0,S→S,6.36,answer,True,0.73,0.45,what is weight of education qualification and experience
2,manual,NHM03,section,list,gate,1.00,1.0,1.0,1.0,1.0,M→M,6.63,scoped,True,0.00,0.00,what are Responsibilities of DDM and DMC
4,manual,NHM05,section,list,context,1.00,1.0,1.0,1.0,1.0,S→S,9.28,answer,True,0.38,0.06,what are Contents of Code of Conduct Policy of any state
5,manual,NHM06,section,list,llm,0.83,1.0,1.0,1.0,1.0,S→S,6.26,answer,True,0.83,0.33,Tools of HR management list them for me
8,manual,NHM09,section,navigation,context,0.50,3.0,4.0,3.0,3.0,S→S,1.57,answer,True,0.00,0.00,can you tell for Appointment and Onboarding which page of pdf I should look
9,manual,NHM10,section,list,gate,1.00,1.0,3.0,1.0,1.0,M→M,-1.45,scoped,True,0.00,0.00,what is process of Recruiment list all the key steps
10,manual,NHM11,section,list,context,0.88,1.0,1.0,1.0,1.0,S→S,8.69,answer,True,0.35,0.35,Job Responsibilities of Specialist – Obstetrics & Gynaecology list them all
11,manual,NHM12,section,list,llm,1.00,1.0,1.0,1.0,1.0,S→S,4.56,answer,True,0.59,0.18,what is P4p of odisha list them all
13,manual,NHM14,section,scenario,llm,0.83,13.0,1.0,3.0,1.0,P→P,-4.22,answer_with_note,True,0.50,0.00,who is going to pay from travel and filed visit


In [29]:
for qid in ["NHM03", "NHM10", "NHM16", "NHM22", "C-038"]:
    item = QUESTIONS[QUESTIONS.qid == qid].iloc[0]
    print(qid, CR.prepare(item.question)["signals"])

NHM03 {'best': 6.63, 'hard_out': ['DMC'], 'soft_out': [], 'missing': []}
NHM10 {'best': -1.45, 'hard_out': ['Recruiment'], 'soft_out': [], 'missing': []}
NHM16 {'best': 5.24, 'hard_out': ['Expalin'], 'soft_out': [], 'missing': []}
NHM22 {'best': 0.26, 'hard_out': ['Ffrom'], 'soft_out': [], 'missing': []}
C-038 {'best': 1.93, 'hard_out': ['ICU'], 'soft_out': [], 'missing': []}


In [30]:
# 14.7 Gate fix check: old vs new corrective layer (no LLM)
import importlib
import src.corrective

OLD_CR = CR
importlib.reload(src.corrective)
NEW_CR = src.corrective.Corrective(R, SpellFixer(R.chunks))
NEGATIVES = pd.read_csv(EVAL.parent / "negatives_50.csv")
NEG_COL = "question"                     # change if your file names the column differently
print("negatives columns:", list(NEGATIVES.columns))

def compare(pairs):
    rows = []
    for qid, question in pairs:
        old, new = OLD_CR.prepare(question), NEW_CR.prepare(question)
        rows.append({"qid": qid, "old": old["action"], "new": new["action"], "changes": new["changes"],
                     "hard_out": new["signals"]["hard_out"], "question": question[:70]})
    return pd.DataFrame(rows)

gate = QUESTIONS[QUESTIONS.qid.isin(["NHM03", "NHM10", "NHM16", "NHM22", "C-038", "NHM15", "C-028"])]
display(compare(zip(gate.qid, gate.question)))
negatives = compare(zip(NEGATIVES.index.astype(str), NEGATIVES[NEG_COL]))
display(pd.crosstab(negatives.old, negatives.new, margins=True))
display(negatives[negatives.old != negatives.new])

negatives columns: ['type', 'question', 'marker', 'marker_in_pdf', 'best', 'zone', 'action', 'not_in_pdf', 'not_in_top5', 'gate_only', 'gate_plus_check']


,qid,old,new,changes,hard_out,question
0,NHM03,scoped,answer,"[(DMC, DCM)]",[],what are Responsibilities of DDM and DMC
1,NHM10,scoped,answer,"[(Recruiment, recruitment)]",[],what is process of Recruiment list all the key steps
2,NHM15,reject,reject,[],[],who is going to pay for the hotel and cab if I want in Filed visit
3,NHM16,scoped,answer,"[(Expalin, explain)]",[],what is Public health care cadre in tamil nadu Expalin them
4,NHM22,scoped,answer,"[(Ffrom, from)]",[],can tell me the sample of exit clearance Ffrom
5,C-028,reject,reject,[hota],[],Staff nurse post ke liye kaunsa test hota hai jo programme management
6,C-038,answer,answer,[],[],Our hospital has no General Medicine specialist. Which specialist's jo


new,answer_with_note,reject,scoped,All
old,,,,
answer_with_note,9,0,0,9
reject,0,25,0,25
scoped,0,0,16,16
All,9,25,16,50


,qid,old,new,changes,hard_out,question


In [33]:
# 14.8 Answer-stage A/B: same evidence, two versions of rule 7 (gpt-oss on Groq, run from the laptop)
from dotenv import load_dotenv
import src.answer as answer_mod
from src.llm import chat, real_answer_check

load_dotenv(ROOT / ".env")                      # GROQ_API_KEY from the gitignored .env
RULE_V1 = "7. Be short: 1-5 sentences or a short list. No extra advice."
RULE_V2 = ("7. Fit the length to the question: one fact -> 1-3 sentences; a list, steps, a form, a letter "
           "or 'all' -> every item as a bullet list. No extra advice.")
PROMPT_V2 = answer_mod.SYSTEM_PROMPT.replace(RULE_V1, RULE_V2)
assert PROMPT_V2 != answer_mod.SYSTEM_PROMPT, "rule 7 text not found in SYSTEM_PROMPT"
ANSWER_MODEL = "groq:openai/gpt-oss-120b"
print("model check:", real_answer_check(ANSWER_MODEL))

rows = []
for item in QUESTIONS[QUESTIONS.source == "manual"].itertuples():
    prep = CR.prepare(item.question)
    if not prep["prompt"]:
        print(f"{item.qid}: {prep['action']} (no LLM call)")
        continue
    for version, system in (("v1", answer_mod.SYSTEM_PROMPT), ("v2", PROMPT_V2)):
        answer = chat(ANSWER_MODEL, system, prep["prompt"])
        rows.append({"qid": item.qid, "version": version, "prompt_cov": coverage(item.expected, prep["prompt"]),
                     "answer_cov": coverage(item.expected, answer), "words": len(answer.split()), "answer": answer})
AB = pd.DataFrame(rows)
AB.to_csv(EVAL / "answer_ab_rule7.csv", index=False)
display(AB.pivot(index="qid", columns="version", values=["answer_cov", "words"]))
display(AB.groupby("version")[["answer_cov", "words"]].mean().round(2))

model check: 42
NHM03: scoped (no LLM call)
NHM07: reject (no LLM call)
NHM08: reject (no LLM call)
NHM10: scoped (no LLM call)
NHM15: reject (no LLM call)
NHM16: scoped (no LLM call)
NHM22: scoped (no LLM call)


answer_cov        words       
version         v1    v2     v1     v2
qid                                   
NHM01         0.43  0.36   75.0   93.0
NHM02         0.36  0.45   45.0   34.0
NHM04         0.75  0.75  191.0  282.0
NHM05         0.19  0.06   18.0   12.0
NHM06         0.33  0.33   89.0   67.0
NHM09         0.00  0.00   28.0   31.0
NHM11         0.35  0.41  268.0  300.0
NHM12         0.18  0.18   71.0   98.0
NHM13         0.55  0.55   78.0   98.0
NHM14         0.00  0.33    5.0  135.0
NHM17         0.33  0.44  100.0  121.0
NHM18         0.16  0.32  105.0   83.0
NHM19         0.09  0.09    8.0    7.0
NHM20         0.00  0.89    5.0  493.0
NHM21         0.00  0.00    5.0    5.0
NHM23         0.38  0.38  152.0  350.0

,answer_cov,words
version,,
v1,0.26,77.69
v2,0.35,138.06


In [34]:
for row in AB[AB.qid.isin(["NHM19", "NHM05", "NHM12"])].itertuples():
    print(f"--- {row.qid} {row.version} ({row.words} words)\n{row.answer}\n")

--- NHM05 v1 (18 words)
The Code of Conduct Policy of a State should detail accepted workplace behaviour, including accepting gifts and entertainment【1】.

--- NHM05 v2 (12 words)
- Accepted behaviour in the workplace relating to **accepting gifts and entertainment**【1】

--- NHM12 v1 (71 words)
- HR rationalisation (re‑structuring of hierarchical slabs) [2]  
- A blended payment system that divides total contractual staff compensation into three parts [5]  
- A performance‑based incentive pool (≈ Rs 60 crore annually for ~4 000 PMU staff and ~11 000 clinical/para‑medical staff) [4]  
- A hierarchical structure with defined job responsibilities and reporting pathways [3]  
- A performance‑management system that links contract renewal and staff compensation to performance grades [3]

--- NHM12 v2 (98 words)
- **HR Rationalisation** – restructuring the hierarchical slabs and remuneration levels for contractual staff [2]  
- **Blended payment system** – total compensation divided into three

In [35]:
# 14.9 Safety check for rule 7 v2: negatives that reach the LLM must still be refused
REFUSAL = ("not found in the guideline", "does not cover", "does not mention", "not covered")
neg_rows = []
for question in NEGATIVES[NEG_COL]:
    prep = NEW_CR.prepare(question)
    if prep["prompt"]:
        answer = chat(ANSWER_MODEL, PROMPT_V2, prep["prompt"])
        neg_rows.append({"question": question[:70], "refused": any(text in answer.lower() for text in REFUSAL),
                         "answer": answer[:200]})
NEG_AB = pd.DataFrame(neg_rows)
display(NEG_AB)
print(f"refused {NEG_AB.refused.sum()} of {len(NEG_AB)}")

,question,refused,answer
0,What is the process for claiming hotel stay expenses for Railways staf,False,"- After the travel, the staff member must prepare a tour report, which becom..."
1,What is the probation period for contractual staff in the Army?,True,Not found in the guideline.
2,Should Amazon review the JD and ToR before advertising a warehouse pos,True,Not found in the guideline.
3,How is the entry level salary decided for a Google consultant?,True,Not found in the guideline.
4,What grievance redressal process should Microsoft employees follow?,True,Not found in the guideline.
5,How should a Bollywood film crew be recruited on contract?,True,Not found in the guideline.
6,What is the EPF contribution rate for private sector employees?,True,Not found in the guideline.
7,How is income tax calculated on a government salary?,True,Not found in the guideline.
8,How do I apply for an Ayushman Bharat card?,True,Not found in the guideline.


refused 8 of 9
